In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, lag, when, lit, concat, sum, monotonically_increasing_id
from pyspark.sql.window import Window


# spark = SparkSession.builder \
#     .appName("ClickStreamSession") \
#     .getOrCreate()

schema = "click_time STRING, user_id STRING"

data = [
    ("2018-01-01 11:00:00", "u1"),
    ("2018-01-01 12:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 14:00:00", "u1"),
    ("2018-01-01 15:00:00", "u1"),
    ("2018-01-01 11:00:00", "u2"),
    ("2018-01-02 11:00:00", "u2")
]

clickstream_df = spark.createDataFrame(data, schema=schema)

In [0]:
clickstream_df = clickstream_df.withColumn("click_time" , unix_timestamp(col("click_time")))

In [0]:
window_spac = Window.partitionBy(col("user_id")).orderBy(col("click_time"))
func = lag(col("click_time")).over(window_spac)

clickstream_df = clickstream_df.withColumn("prev_time" , func).withColumn("time_diff" , col("click_time") - col("prev_time"))
clickstream_df.show()
# func = min("click_time").over(window_spac)
# clickstream_df = clickstream_df.withColumn("min_time" , func )

+----------+-------+----------+---------+------------+-----------+
|click_time|user_id| prev_time|time_diff|isNewSession|session_num|
+----------+-------+----------+---------+------------+-----------+
|1514804400|     u1|      NULL|     NULL|         yes|       1_u1|
|1514808000|     u1|1514804400|     3600|         yes|       2_u1|
|1514811600|     u1|1514808000|     3600|         yes|       3_u1|
|1514811600|     u1|1514811600|        0|          no|       3_u1|
|1514815200|     u1|1514811600|     3600|         yes|       4_u1|
|1514818800|     u1|1514815200|     3600|         yes|       5_u1|
|1514804400|     u2|      NULL|     NULL|         yes|       1_u2|
|1514890800|     u2|1514804400|    86400|         yes|       2_u2|
+----------+-------+----------+---------+------------+-----------+



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-8378249239156053>, line 6
      4 clickstream_df = clickstream_df.withColumn("prev_time" , func).withColumn("time_diff" , col("click_time") - col("prev_time"))
      5 clickstream_df.show()
----> 6 func = min("click_time").over(window_spac)
      7 clickstream_df = clickstream_df.withColumn("min_time" , func )

AttributeError: 'str' object has no attribute 'over'

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import min

user_window = Window.partitionBy("user_id")

clickstream_df = clickstream_df.withColumn(
    "min_time",
    min("click_time").over(user_window)
)
clickstream_df.show()

+----------+-------+----------+---------+------------+-----------+----------+
|click_time|user_id| prev_time|time_diff|isNewSession|session_num|  min_time|
+----------+-------+----------+---------+------------+-----------+----------+
|1514804400|     u1|      NULL|     NULL|         yes|       1_u1|1514804400|
|1514808000|     u1|1514804400|     3600|         yes|       2_u1|1514804400|
|1514811600|     u1|1514808000|     3600|         yes|       3_u1|1514804400|
|1514811600|     u1|1514811600|        0|          no|       3_u1|1514804400|
|1514815200|     u1|1514811600|     3600|         yes|       4_u1|1514804400|
|1514818800|     u1|1514815200|     3600|         yes|       5_u1|1514804400|
|1514804400|     u2|      NULL|     NULL|         yes|       1_u2|1514804400|
|1514890800|     u2|1514804400|    86400|         yes|       2_u2|1514804400|
+----------+-------+----------+---------+------------+-----------+----------+



In [0]:
clickstream_df = clickstream_df.withColumn(
    "isNewSession",
    when(
        col("time_diff").isNull(),
        lit("yes")
    )
    .when(
        col("time_diff") > 1800,
        lit("yes")
    )
    .when(
       col("click_time") - col("min_time") >= 7200 ,
       lit("yes")
    )
    .otherwise(lit("no"))
)


In [0]:
clickstream_df.show()

+----------+-------+----------+---------+------------+-----------+----------+
|click_time|user_id| prev_time|time_diff|isNewSession|session_num|  min_time|
+----------+-------+----------+---------+------------+-----------+----------+
|1514804400|     u1|      NULL|     NULL|         yes|       1_u1|1514804400|
|1514808000|     u1|1514804400|     3600|         yes|       2_u1|1514804400|
|1514811600|     u1|1514808000|     3600|         yes|       3_u1|1514804400|
|1514811600|     u1|1514811600|        0|         yes|       3_u1|1514804400|
|1514815200|     u1|1514811600|     3600|         yes|       4_u1|1514804400|
|1514818800|     u1|1514815200|     3600|         yes|       5_u1|1514804400|
|1514804400|     u2|      NULL|     NULL|         yes|       1_u2|1514804400|
|1514890800|     u2|1514804400|    86400|         yes|       2_u2|1514804400|
+----------+-------+----------+---------+------------+-----------+----------+



In [0]:

Window_spec = Window.partitionBy(col("user_id")).orderBy(col("click_time"))
# func = sum()
clickstream_df =clickstream_df.withColumn("session_num" ,sum( when(col("isNewSession")=="yes" , lit(1)).otherwise(lit(0))).over(Window_spec) ).withColumn("session_num" , concat("session_num" , lit("_") , col("user_id")))
clickstream_df.show()

+----------+-------+----------+---------+------------+-----------+----------+
|click_time|user_id| prev_time|time_diff|isNewSession|session_num|  min_time|
+----------+-------+----------+---------+------------+-----------+----------+
|1514804400|     u1|      NULL|     NULL|         yes|       1_u1|1514804400|
|1514808000|     u1|1514804400|     3600|         yes|       2_u1|1514804400|
|1514811600|     u1|1514808000|     3600|         yes|       4_u1|1514804400|
|1514811600|     u1|1514811600|        0|         yes|       4_u1|1514804400|
|1514815200|     u1|1514811600|     3600|         yes|       5_u1|1514804400|
|1514818800|     u1|1514815200|     3600|         yes|       6_u1|1514804400|
|1514804400|     u2|      NULL|     NULL|         yes|       1_u2|1514804400|
|1514890800|     u2|1514804400|    86400|         yes|       2_u2|1514804400|
+----------+-------+----------+---------+------------+-----------+----------+

